# Matrizes de confusão por classificador

Gera figuras de matriz de confusão para os **6 classificadores** (modelos afinados).

- Treino: conjunto de desenvolvimento (sem hold-out)
- Teste: hold-out
- Predição: máscara top-4 por score

Como o problema é multi-rótulo (13 canais), mostramos:
1. matriz **agregada** 2×2 (todos os FBGs empilhados);
2. grade 2×2 **por FBG** (por classificador).

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.classifiers import predict_topk_mask
from src.cv_utils import load_cv_splits
from src.data_utils import FIGURES_DIR, K_DEFAULT, RESULTS_DIR, load_prepared_dataset
from src.tuned_models import build_tuned_classifiers

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = FIGURES_DIR / "confusion"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Figuras em: {OUT_DIR}")

## 1. Dados e splits

In [ ]:
data = load_prepared_dataset()
X = np.asarray(data["X"], dtype=float)
y = np.asarray(data["y_mask"], dtype=int)

splits = load_cv_splits()
dev = np.asarray(splits["dev_idx"], dtype=int).ravel()
ho = np.asarray(splits["holdout_idx"], dtype=int).ravel()
assert len(np.intersect1d(dev, ho)) == 0

X_tr, y_tr = X[dev], y[dev]
X_te, y_te = X[ho], y[ho]
print(f"Treino (dev): {X_tr.shape[0]} | Teste (hold-out): {X_te.shape[0]} | k={K_DEFAULT}")

## 2. Treinar e predizer

In [ ]:
models = build_tuned_classifiers()
preds: dict[str, np.ndarray] = {}

for name, est in models.items():
    print(f"Treinando {name}...")
    est.fit(X_tr, y_tr)
    preds[name] = predict_topk_mask(est, X_te, k=K_DEFAULT)
    assert preds[name].shape == y_te.shape
    assert np.all(preds[name].sum(axis=1) == K_DEFAULT)
    print(f"  OK — shape {preds[name].shape}")

## 3. Matriz agregada 2×2 (por classificador)

Empilha os 13 canais: cada célula (amostra, FBG) conta como uma decisão binária.

In [ ]:
def plot_agg_confusion(y_true: np.ndarray, y_pred: np.ndarray, title: str, out_path: Path):
    cm = confusion_matrix(y_true.ravel(), y_pred.ravel(), labels=[0, 1])
    fig, ax = plt.subplots(figsize=(4.2, 3.6))
    ConfusionMatrixDisplay(cm, display_labels=["0 (fora)", "1 (máscara)"]).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format="d"
    )
    ax.set_title(title)
    ax.set_xlabel("Predito")
    ax.set_ylabel("Verdadeiro")
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.show()
    plt.close(fig)
    tn, fp, fn, tp = cm.ravel()
    return {"TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)}

rows = []
for name, y_pred in preds.items():
    stats = plot_agg_confusion(
        y_te,
        y_pred,
        title=f"{name} — confusão agregada (hold-out)",
        out_path=OUT_DIR / f"cm_agg_{name}.png",
    )
    stats["classifier"] = name
    rows.append(stats)

summary = pd.DataFrame(rows)[["classifier", "TN", "FP", "FN", "TP"]]
summary.to_csv(RESULTS_DIR / "confusion_agg_holdout.csv", index=False)
summary

## 4. Matrizes 2×2 por FBG (grade)

Uma figura por classificador, com 13 subplots (FBG 0–12).

In [ ]:
def plot_per_fbg_grid(y_true: np.ndarray, y_pred: np.ndarray, title: str, out_path: Path):
    n_fbgs = y_true.shape[1]
    ncols = 5
    nrows = int(np.ceil(n_fbgs / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 2.4 * nrows))
    axes = np.atleast_2d(axes)

    for j in range(n_fbgs):
        r, c = divmod(j, ncols)
        ax = axes[r, c]
        cm = confusion_matrix(y_true[:, j], y_pred[:, j], labels=[0, 1])
        ConfusionMatrixDisplay(cm, display_labels=["0", "1"]).plot(
            ax=ax, cmap="Blues", colorbar=False, values_format="d"
        )
        ax.set_title(f"FBG {j}", fontsize=9)
        ax.set_xlabel("")
        ax.set_ylabel("")

    for j in range(n_fbgs, nrows * ncols):
        r, c = divmod(j, ncols)
        axes[r, c].axis("off")

    fig.suptitle(title, y=1.01)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

for name, y_pred in preds.items():
    plot_per_fbg_grid(
        y_te,
        y_pred,
        title=f"{name} — confusão por FBG (hold-out)",
        out_path=OUT_DIR / f"cm_per_fbg_{name}.png",
    )

## 5. Painel comparativo (só agregadas)

Todos os classificadores lado a lado para comparação rápida.

In [ ]:
names = list(preds.keys())
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
axes = axes.ravel()

for ax, name in zip(axes, names):
    cm = confusion_matrix(y_te.ravel(), preds[name].ravel(), labels=[0, 1])
    ConfusionMatrixDisplay(cm, display_labels=["0", "1"]).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format="d"
    )
    ax.set_title(name)
    ax.set_xlabel("Predito")
    ax.set_ylabel("Verdadeiro")

fig.suptitle("Matrizes agregadas — hold-out (top-4)", y=1.02)
fig.tight_layout()
panel_path = OUT_DIR / "cm_agg_all_classifiers.png"
fig.savefig(panel_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Salvo: {panel_path}")
print(f"Todas as figuras: {OUT_DIR}")